# FSOCO cone detection - Colab training
Free T4 GPU is enough for YOLO11n. **Runtime > Change runtime type > T4 GPU** first.

Steps: (1) get the code, (2) download FSOCO (~24 GB, official source), (3) convert, (4) train, (5) evaluate, (6) demo.
Tip: use a Google Drive folder for `WORK` so results survive a disconnected session.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/fsoco_cones'   # results + dataset cache live here
!mkdir -p {WORK}

## 1. Code
Replace the URL with your GitHub repo (or upload the project folder / zip to Colab).

In [ ]:
REPO = 'https://github.com/<your-user>/fsoco-cone-detection.git'
!git clone {REPO} /content/fsoco-cone-detection
%cd /content/fsoco-cone-detection
!pip install -q -r requirements.txt

## 2. Download FSOCO (bounding boxes, ~24 GB)
Official page: https://fsoco.github.io/fsoco-dataset/download - no registration needed.
Downloading to Drive once saves you from repeating it every session (needs ~50 GB free Drive space while unpacking;
if short on space, download to `/content` instead - it is lost when the runtime resets).

In [ ]:
import os
DL = f'{WORK}/fsoco_boxes'
os.makedirs(DL, exist_ok=True)
URL = 'http://fsoco.cs.uni-freiburg.de/src/download_boxes_train.php'
# The server may answer with a redirect/zip; -J -O keeps the server-provided file name.
!cd {DL} && wget --content-disposition -c "{URL}"
!ls -lh {DL}

In [ ]:
# Unpack (adjust the file name printed above if it differs)
!cd {DL} && for f in *.zip; do unzip -q -n "$f"; done; ls | head
# SRC must be the folder that contains the team folders (each with img/ and ann/)

## 3. Convert to YOLO format (quick 400-image subset first, then the full set)

In [ ]:
SRC = DL   # change if the archive unpacked into a sub-folder
!python data_prep.py --src {SRC} --out /content/data/fsoco_small --subset 400 --clean
!python train.py --data /content/data/fsoco_small/data.yaml --epochs 3 --batch 16 --name smoke --project /content/runs

In [ ]:
!python data_prep.py --src {SRC} --out /content/data/fsoco_yolo --oversample --split-by team --clean

## 4. Train (about 1-2 h on a T4 for 100 epochs)

In [ ]:
!python train.py --data /content/data/fsoco_yolo/data.yaml --epochs 100 --batch 32 --project {WORK}/runs --name fsoco_yolo11n

## 5. Evaluate

In [ ]:
W = f'{WORK}/runs/fsoco_yolo11n/weights/best.pt'
!python evaluate.py --weights {W} --data /content/data/fsoco_yolo/data.yaml --split test --out {WORK}/results
from IPython.display import Image, display
display(Image(f'{WORK}/results/confusion_matrix_normalized.png'))
display(Image(f'{WORK}/results/predictions_grid.jpg'))

## 6. Demo on your own video + tracking

In [ ]:
from google.colab import files
up = files.upload()            # choose a driving video
VID = next(iter(up))
!python inference.py --weights {W} --source {VID} --out {WORK}/detect.mp4 --gif {WORK}/detect.gif
!python tracking.py  --weights {W} --source {VID} --out {WORK}/track.mp4  --gif {WORK}/track.gif
from IPython.display import Image
Image(f'{WORK}/track.gif')